# Week 2 Lab｜Derivatives & Differentiation Rules

**Goal**: make Week-2 theory *run and check itself* in Python —
1. **Numerical derivative** (central difference) vs the exact value
2. **Verify every rule** (power / product / quotient / chain / trig / exp / log / log-diff) with **SymPy**
3. **Implicit differentiation**, visualized: the tangent to `x^2 + y^2 = 1` has slope `-x/y`
4. **Mini-autodiff**: a tiny `Value` class with `+`, `*`, forward & **backward** (backprop, micrograd spirit), cross-checked against SymPy

**How to run**: upload to [Google Colab](https://colab.research.google.com/) or local Jupyter, run top to bottom.
Cells marked `# TODO 學生練習` are for you to fill in.

> 圖表標籤用英文/數學符號以避免中文變豆腐字;中文說明都放在文字格與註解。

In [ ]:
# === Setup (run me first) ===
import math
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True
print("environment ready | numpy", np.__version__, "| sympy", sp.__version__)

### (optional) Chinese labels on plots

預設圖表用英文標籤,避免中文變「豆腐字」。若你在 Colab 想要中文座標/標題,
把下一格的註解取消再執行(只需一次)。

In [ ]:
# 想要中文圖標時,取消以下註解執行(Colab 適用;本機 Jupyter 需自備 CJK 字型)
# !apt-get -qq install fonts-noto-cjk > /dev/null
# import matplotlib
# matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# matplotlib.rcParams['axes.unicode_minus'] = False
print("default: English labels; uncomment above for Chinese")

## Lab 1｜Numerical derivative (central difference)

導數的定義是 `(f(x+h)-f(x))/h` 取極限。數值上,**central difference**
`(f(x+h) - f(x-h)) / (2h)` 收斂更快(誤差 `O(h^2)`)。拿它跟手算的理論導數對一對。

In [ ]:
def numerical_derivative(f, x, h=1e-5):
    """Central difference approximation of f'(x). Error ~ O(h^2)."""
    return (f(x + h) - f(x - h)) / (2 * h)

# compare numerical vs theoretical at one sample point each
tests = [
    ("x^2   -> 2x",           lambda x: x**2,  lambda x: 2*x,                  3.0),
    ("sin x -> cos x",        math.sin,        math.cos,                       1.0),
    ("e^x   -> e^x",          math.exp,        math.exp,                       0.5),
    ("x^x   -> x^x(ln x+1)",  lambda x: x**x,  lambda x: x**x*(math.log(x)+1), 2.0),
]
print(f"{'function':22} {'numerical':>15} {'theory':>15} {'abs err':>11}")
for name, f, fp, x0 in tests:
    approx, exact = numerical_derivative(f, x0), fp(x0)
    print(f"{name:22} {approx:15.8f} {exact:15.8f} {abs(approx-exact):11.2e}")

In [ ]:
# and visually: numerical d/dx sin x should land on cos x
xs  = np.linspace(-2*np.pi, 2*np.pi, 400)
num = (np.sin(xs + 1e-5) - np.sin(xs - 1e-5)) / (2e-5)
plt.plot(xs, num, lw=4, alpha=0.35, label="numerical d/dx sin x")
plt.plot(xs, np.cos(xs), 'r--', label="cos x (exact)")
plt.title("Central-difference derivative of sin x matches cos x")
plt.legend(); plt.show()

In [ ]:
# TODO 學生練習:用 numerical_derivative 驗證 d/dx tan(x) = sec^2(x) 在 x = 1
# 提示:sec^2(x) = 1 / cos(x)**2
# print(numerical_derivative(math.tan, 1.0), 1/math.cos(1.0)**2)

## Lab 2｜Verify the rules with SymPy

手算的每條法則,交給 `sympy.diff` 當「標準答案」核對。看到 SymPy 的輸出和你紙上的一致,就放心了。

In [ ]:
x = sp.symbols('x')

rules = {
    "Power      d/dx x^5":       x**5,
    "Sum/Diff   d/dx 3x^4-5x^2+7": 3*x**4 - 5*x**2 + 7,
    "Product    d/dx x^2 e^x":   x**2*sp.exp(x),
    "Quotient   d/dx x/(x^2+1)": x/(x**2 + 1),
    "Chain      d/dx (x^2+1)^3": (x**2 + 1)**3,
    "sin/cos    d/dx sin x":     sp.sin(x),
    "tan        d/dx tan x":     sp.tan(x),
    "exp        d/dx e^x":       sp.exp(x),
    "log        d/dx ln x":      sp.log(x),
    "log-diff   d/dx x^x":       x**x,
}
for name, f in rules.items():
    print(f"{name:26} = {sp.simplify(sp.diff(f, x))}")

In [ ]:
# cross-check a hand answer: d/dx x/(x^2+1)  ==  (1 - x^2)/(x^2+1)^2 ?
hand = (1 - x**2) / (x**2 + 1)**2
print("quotient-rule hand answer correct:",
      sp.simplify(sp.diff(x/(x**2 + 1), x) - hand) == 0)

# implicit: d/dx of x^x again, but via log-diff by hand -> x^x*(ln x + 1)
print("log-diff hand answer correct:",
      sp.simplify(sp.diff(x**x, x) - x**x*(sp.log(x) + 1)) == 0)

## Lab 3｜Implicit differentiation, visualized

圓 `x^2 + y^2 = 1` 沒法簡單寫成 `y = f(x)`。隱函數微分給出切線斜率 `y' = -x/y`。
畫出圓與某點的切線,順便驗證「切線 ⟂ 半徑」。

In [ ]:
# unit circle x^2 + y^2 = 1 ; implicit diff => y' = -x/y
theta = np.linspace(0, 2*np.pi, 400)
plt.plot(np.cos(theta), np.sin(theta), label="x^2 + y^2 = 1")

px, py = math.cos(math.pi/4), math.sin(math.pi/4)   # a point on the circle (45 deg)
slope  = -px / py                                    # y' = -x/y
tx = np.linspace(px - 0.8, px + 0.8, 10)
plt.plot(tx, py + slope*(tx - px), 'r--', label=f"tangent, slope = {slope:.2f}")
plt.plot([0, px], [0, py], 'g:', label="radius")
plt.plot(px, py, 'ko')
plt.gca().set_aspect('equal'); plt.legend(loc='upper right')
plt.title("Implicit diff: tangent slope = -x/y"); plt.show()

print(f"point = ({px:.3f}, {py:.3f})   slope -x/y = {slope:.4f}")
print("tangent-perp-radius check (should be -1):", slope * (py/px))

In [ ]:
# TODO 學生練習:對橢圓 x^2/4 + y^2 = 1 隱函數微分,求 y',並在點 (0,1) 或 (2,0) 討論斜率
# 手算提示:2x/4 + 2y*y' = 0 => y' = -x/(4y)

## Lab 4｜Mini-autodiff — the chain rule, automated

鏈鎖法則 = **backpropagation** 的核心。這裡手刻一個極簡版(micrograd 精神):
一個純量 `Value` 節點,支援 `+`、`*`,`forward` 記住怎麼算出來、`backward` 用鏈鎖把梯度**反向**傳回去。

In [ ]:
class Value:
    """A minimal scalar autodiff node (micrograd 精神). Supports + and *, plus backward()."""
    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None      # local backprop closure
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():                   # d(out)/d(self)=1, d(out)/d(other)=1
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():                   # product rule, locally
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    __radd__ = __add__
    __rmul__ = __mul__

    def backward(self):
        # 1) topological order of the graph, 2) apply chain rule in reverse
        topo, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0                    # d(output)/d(output) = 1
        for v in reversed(topo):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

print("Value class ready")

In [ ]:
# forward:  f = x*y + x*x   at x=3, y=4   (expect f = 12 + 9 = 21)
x = Value(3.0)
y = Value(4.0)
f = x*y + x*x
f.backward()

print("forward  f =", f.data)
print("df/dx (autodiff) =", x.grad, "  expected y + 2x = 10")
print("df/dy (autodiff) =", y.grad, "  expected x     = 3")

# cross-check against SymPy
sx, sy = sp.symbols('sx sy')
expr = sx*sy + sx*sx
print("df/dx (sympy)    =", sp.diff(expr, sx).subs({sx: 3, sy: 4}))
print("df/dy (sympy)    =", sp.diff(expr, sy).subs({sx: 3, sy: 4}))

**看懂了嗎?** `backward()` 做的就是:把整張運算圖做拓撲排序,從輸出的梯度 `1` 出發,
用每個節點的**區域導數**(加法傳 `1`、乘法傳對方的值)沿鏈鎖**反向相乘累加**。
把 `+`、`*` 換成矩陣乘與非線性函數、把幾個 `Value` 換成幾百萬個參數,就是深度學習的 backprop。
第 6 週我們會用它訓練一個小模型。

In [ ]:
# TODO 學生練習:自己搭一個式子,例如 g = (x + y) * x,在 x=2, y=5 求 dg/dx, dg/dy
# 先手算(dg/dx = 2x + y = 9, dg/dy = x = 2),再用 Value 驗證
# x = Value(2.0); y = Value(5.0); g = (x + y) * x; g.backward(); print(x.grad, y.grad)

## 收尾 · 與筆試的連結

| 這個 Lab | 對應觀念 |
|---|---|
| Lab 1 central-difference | 導數定義、切線斜率(觀念 1、2) |
| Lab 2 SymPy 驗證各法則 | 冪/常數和差/積/商/三角/指/對/鏈鎖/對數微分(觀念 4–12、14) |
| Lab 3 圓的切線斜率 | 隱函數微分(觀念 13) |
| Lab 4 mini-autodiff | 鏈鎖法則 = backpropagation(觀念 12) |

### 進階徽章(選做)
1. 把 `numerical_derivative` 的 `h` 從 `1e-1` 掃到 `1e-10`,畫出誤差對 `h` 的 log-log 圖(會看到先降後升:截斷誤差 vs 浮點誤差)。
2. 給 `Value` 加上 `__pow__`(冪)或 `relu()`,讓 mini-autodiff 能表示更多函數。
3. 用 Lab 4 的 `Value` 手算 `f = x*x*x`(即 `x^3`)在 `x=2` 的導數,對照 `d/dx x^3 = 3x^2 = 12`。